# L'Oreal India Job Scraper
## India Jobs Scraper (v2.1 - March 2026)
**Source:** careers.loreal.com

**ATS Detection:** Automatic API + Selenium fallback

In [1]:
!pip install selenium webdriver-manager pandas openpyxl requests beautifulsoup4 lxml playwright -q


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import sys, time, random, re
from pathlib import Path

# Add scripts dir to path so we can import scraper_utils
SCRIPTS_DIR = Path.home() / "Job_Scrapers" / "All_Scripts"
sys.path.insert(0, str(SCRIPTS_DIR))

from scraper_utils import *
import requests
from bs4 import BeautifulSoup
from datetime import datetime, timedelta

# ── LOCATION CONFIG ──────────────────────────────────────────────────────────
# Change to "" to scrape globally (all countries).
# The matching pipeline's pre-filter handles India-specific narrowing.
# Set to "India" here only if you want to reduce volume at scrape time.
LOCATION_FILTER = ""
COUNTRY_CODE   = ""  # e.g. "in" for SmartRecruiters country= param; "" = all
# ─────────────────────────────────────────────────────────────────────────────

print("Imports loaded. Date:", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print(f"Location filter: '{LOCATION_FILTER}' (empty = broad/global scraping)")


scraper_utils.py loaded successfully
Schema: 25 columns
Skills DB: 79 skills
Imports loaded. Date: 2026-03-31 23:45:12
Location filter: '' (empty = broad/global scraping)


In [3]:
COMPANY = "Loreal"
OUTPUT_DIR = get_output_dir(COMPANY)
print(f"Output directory: {OUTPUT_DIR}")


Output directory: /Users/incognito/Job_Scrapers/All_CSV_Outputs/Loreal/Outputs/2026_03_31


In [4]:
print("=" * 60)
print("L\'OREAL INDIA JOB SCRAPER")
print("Source: careers.loreal.com (Phenom platform)")
print("=" * 60)

from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC


def fetch_jd_selenium(driver, url, timeout=10):
    """Visit a job detail page and extract the JD text."""
    try:
        driver.get(url)
        time.sleep(random.uniform(2, 4))
        soup = BeautifulSoup(driver.page_source, "lxml")
        # Try common JD container selectors
        for sel in ["[class*='job-description']", "[class*='jd-info']", "[class*='description']",
                     "[class*='details']", "article", "main", ".content"]:
            el = soup.select_one(sel)
            if el and len(el.get_text(strip=True)) > 100:
                return el.get_text(" ", strip=True)
        # Fallback: get body text
        body = soup.select_one("body")
        return body.get_text(" ", strip=True)[:5000] if body else ""
    except Exception as e:
        print(f"    [WARN] JD fetch failed for {url}: {e}")
        return ""

def fetch_jd_requests(session, url):
    """Fetch a job detail page via requests and extract JD text."""
    try:
        resp = session.get(url, timeout=20)
        if resp.status_code == 200:
            soup = BeautifulSoup(resp.text, "lxml")
            for sel in ["[class*='job-description']", "[class*='jd-info']", "[class*='description']",
                         "[class*='details']", "article", "main"]:
                el = soup.select_one(sel)
                if el and len(el.get_text(strip=True)) > 100:
                    return el.get_text(" ", strip=True)
            body = soup.select_one("body")
            return body.get_text(" ", strip=True)[:5000] if body else ""
    except:
        pass
    return ""


loreal_jobs = []

# Phenom platform uses internal XHR APIs. Try to find them via Selenium.
driver = setup_selenium()
try:
    # Enable network logging to capture XHR API calls
    driver.get("https://careers.loreal.com/global/en/search-results?keywords=&location=India")
    time.sleep(12)

    # Wait for Phenom to render job cards
    try:
        WebDriverWait(driver, 25).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "[data-ph-at-id], [class*=\'job-card\'], [class*=\'search-result\'], a[href*=\'/job/\']"))
        )
    except:
        print("  Waiting longer for Phenom to render...")
        time.sleep(10)

    for page in range(10):
        soup = BeautifulSoup(driver.page_source, "lxml")

        # Remove nav/header/footer
        for unwanted in soup.select("nav, header, footer, [role=\'navigation\']"):
            unwanted.decompose()

        # Phenom career sites typically use data-ph-at-id attributes for job cards
        cards = soup.select("[data-ph-at-id*=\'job\'], [class*=\'job-card\'], [class*=\'search-result-item\']")
        if not cards:
            # Fallback: look for job links
            job_links = soup.select("a[href*=\'/job/\'], a[href*=\'/en/job/\']")
            seen_parents = set()
            for link in job_links:
                parent = link.parent
                if parent and id(parent) not in seen_parents:
                    cards.append(parent)
                    seen_parents.add(id(parent))

        new_count = 0
        for card in cards:
            title_el = card.select_one("h2, h3, h4, [class*=\'title\'], [data-ph-at-id*=\'title\'], a[href*=\'/job/\']")
            title = title_el.get_text(strip=True) if title_el else ""
            loc_el = card.select_one("[class*=\'location\'], [class*=\'city\'], [data-ph-at-id*=\'location\']")
            loc = loc_el.get_text(strip=True) if loc_el else "India"
            dept_el = card.select_one("[class*=\'department\'], [class*=\'category\'], [data-ph-at-id*=\'department\']")
            dept = dept_el.get_text(strip=True) if dept_el else ""

            link = card.select_one("a[href*=\'/job/\']")
            href = link.get("href", "") if link else ""

            if is_valid_job_title(title) and title not in [j["title"] for j in loreal_jobs]:
                full_url = href if href.startswith("http") else f"https://careers.loreal.com{href}" if href else ""
                loreal_jobs.append({
                    "job_id": href.split("/")[-1] if href else str(len(loreal_jobs)),
                    "title": title,
                    "company_name": "L\'Oreal",
                    "raw_jd_text": "",
                    "location_city": loc.split(",")[0].strip(),
                    "industry": "FMCG / Beauty & Cosmetics",
                    "date_posted": datetime.now().strftime("%Y-%m-%d"),
                    "is_active": True,
                    "job_url": full_url,
                    "business_unit": dept,
                    "source_platform": "L\'Oreal Phenom",
                })
                new_count += 1

        print(f"  Page {page+1}: {new_count} new jobs (total: {len(loreal_jobs)})")
        if new_count == 0 and page > 0:
            break

        # Pagination: Phenom uses "Load more" or numbered pages
        try:
            next_btn = driver.find_element(By.CSS_SELECTOR,
                "button[data-ph-at-id*=\'load-more\'], a[aria-label*=\'Next\'], a[aria-label*=\'next\'], [class*=\'next\'] a, button[class*=\'load-more\']")
            driver.execute_script("arguments[0].click();", next_btn)
            time.sleep(4)
        except:
            # Try scrolling for infinite scroll
            driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
            time.sleep(3)
            new_soup = BeautifulSoup(driver.page_source, "lxml")
            new_cards = new_soup.select("[data-ph-at-id*=\'job\'], a[href*=\'/job/\']")
            if len(new_cards) <= len(cards):
                break

    # Fetch JDs for found jobs
    if loreal_jobs:
        print(f"\n  Fetching JD details for up to 30 jobs...")
        for i, job in enumerate(loreal_jobs[:30]):
            if job.get("raw_jd_text") and len(job["raw_jd_text"]) > 100:
                continue
            if job["job_url"]:
                jd = fetch_jd_selenium(driver, job["job_url"])
                if jd:
                    loreal_jobs[i]["raw_jd_text"] = jd
            if (i + 1) % 10 == 0:
                print(f"    Fetched {i+1}/{min(30, len(loreal_jobs))} JDs")

except Exception as e:
    print(f"  Error: {e}")
    import traceback; traceback.print_exc()
finally:
    driver.quit()

print(f"Total L\'Oreal India jobs: {len(loreal_jobs)}")


L'OREAL INDIA JOB SCRAPER
Source: careers.loreal.com (Phenom platform)


  Waiting longer for Phenom to render...


  Page 1: 0 new jobs (total: 0)


Total L'Oreal India jobs: 0


In [5]:
df_loreal = save_results(loreal_jobs, "L'Oreal", OUTPUT_DIR)
if df_loreal is not None:
    print(f"\nSample jobs:")
    cols = ["title","location_city","seniority_level","business_unit","job_url"]
    cols = [c for c in cols if c in df_loreal.columns]
    print(df_loreal[cols].head(10).to_string())


  [WARN] No jobs found for L'Oreal
